# Clinical NLP — Data Preprocessing Pipeline

**Person 1 — Data & Preprocessing**

This notebook preprocesses all 4 datasets and produces ready-to-use files for the team:

| Dataset | Source | Output | Used by |
|---------|--------|--------|---------|
| BC5CDR | HuggingFace (`tner/bc5cdr`) | `ner/bc5cdr_*.conll` | NER person |
| GENIA | Kaggle | `ner/genia_*.conll` | NER person |
| DrugBank | Kaggle | `relations/drugbank_relations.csv` | Relation Extraction person |
| MTSamples | Kaggle | `unlabeled/mtsamples_sentences.txt` | ASR demo person |

**How to run on Kaggle:**
1. Add these Kaggle datasets to your notebook:
   - `nishanthsalian/genia-biomedical-event-dataset`
   - `sergeguillemart/drugbank` (or `devildev89/drug-bank-5110`)
   - `tboyle10/medicaltranscriptions`
2. Run all cells top to bottom
3. Download the `/kaggle/working/processed_data/` folder and share with the team (Google Drive / GitHub)

## 0. Setup

In [ ]:
# Install required packages
!pip install -q datasets transformers spacy scikit-learn tqdm
!python -m spacy download en_core_web_sm -q

In [ ]:
import os
import re
import json
import random
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import spacy

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Output directories ────────────────────────────────────────────────────────
# On Kaggle, /kaggle/working/ is the writable output folder.
# Locally, change this to any path you prefer.
BASE_OUT = Path("/kaggle/working/processed_data")
(BASE_OUT / "ner").mkdir(parents=True, exist_ok=True)
(BASE_OUT / "relations").mkdir(parents=True, exist_ok=True)
(BASE_OUT / "unlabeled").mkdir(parents=True, exist_ok=True)

print("Output directory:", BASE_OUT)
print("Directories created:", list(BASE_OUT.iterdir()))

---
## 1. BC5CDR — Primary NER Dataset

**Source:** HuggingFace `tner/bc5cdr`  
**Entities:** Chemical (→ DRUG), Disease (→ DISEASE)  
**Format:** Already in IOB/CoNLL. We just remap label names to our project schema.  
**Output:** `bc5cdr_train.conll`, `bc5cdr_val.conll`, `bc5cdr_test.conll`

In [ ]:
from datasets import load_dataset

# ── Label remapping: align BC5CDR labels with our project schema ──────────────
BC5CDR_LABEL_MAP = {
    "O":         "O",
    "B-Chemical": "B-DRUG",
    "I-Chemical": "I-DRUG",
    "B-Disease":  "B-DISEASE",
    "I-Disease":  "I-DISEASE",
}

def save_conll(split_data, label_names, label_map, output_path):
    """
    Save a HuggingFace NER split to CoNLL format.
    Each line: token<TAB>label
    Sentences separated by a blank line.
    """
    written, skipped = 0, 0
    with open(output_path, "w", encoding="utf-8") as f:
        for example in tqdm(split_data, desc=f"Writing {Path(output_path).name}"):
            tokens = example["tokens"]
            tag_ids = example["tags"]
            if not tokens:
                skipped += 1
                continue
            for token, tag_id in zip(tokens, tag_ids):
                original = label_names[tag_id]
                mapped   = label_map.get(original, original)
                f.write(f"{token}\t{mapped}\n")
            f.write("\n")   # blank line = sentence boundary
            written += 1
    print(f"  Saved {written} sentences ({skipped} skipped) → {output_path}")

print("Loading BC5CDR from HuggingFace...")
bc5cdr = load_dataset("tner/bc5cdr")

label_names = bc5cdr["train"].features["tags"].feature.names
print(f"Label scheme: {label_names}")
print(f"Splits: { {k: len(v) for k, v in bc5cdr.items()} }")

In [ ]:
save_conll(bc5cdr["train"],      label_names, BC5CDR_LABEL_MAP, BASE_OUT/"ner/bc5cdr_train.conll")
save_conll(bc5cdr["validation"], label_names, BC5CDR_LABEL_MAP, BASE_OUT/"ner/bc5cdr_val.conll")
save_conll(bc5cdr["test"],       label_names, BC5CDR_LABEL_MAP, BASE_OUT/"ner/bc5cdr_test.conll")

print("\nBC5CDR ✓")

---
## 2. GENIA — Supplementary Biomedical NER

**Source:** Kaggle `nishanthsalian/genia-biomedical-event-dataset`  
**Kaggle path:** `/kaggle/input/genia-biomedical-event-dataset/`  
**Entities:** protein (→ PROTEIN), DNA (→ DNA), RNA (→ RNA), cell_line, cell_type  
**Output:** `genia_train.conll`, `genia_val.conll`, `genia_test.conll`

> The code below auto-detects whether the dataset is in CoNLL (plain text) or CSV format.

In [ ]:
# ── Inspect what files are available ─────────────────────────────────────────
GENIA_DIR = Path("/kaggle/input/genia-biomedical-event-dataset")
all_files = list(GENIA_DIR.rglob("*"))
print("Files found:")
for f in all_files:
    print(" ", f)

In [ ]:
# ── GENIA label mapping: map to our schema ────────────────────────────────────
# We keep biomedical entity types as separate classes.
# The NER model can learn these alongside DRUG/DISEASE.
GENIA_LABEL_MAP = {
    "O":            "O",
    "B-protein":    "B-PROTEIN",
    "I-protein":    "I-PROTEIN",
    "B-DNA":        "B-DNA",
    "I-DNA":        "I-DNA",
    "B-RNA":        "B-RNA",
    "I-RNA":        "I-RNA",
    "B-cell_line":  "B-CELL_LINE",
    "I-cell_line":  "I-CELL_LINE",
    "B-cell_type":  "B-CELL_TYPE",
    "I-cell_type":  "I-CELL_TYPE",
    # Some versions use these capitalizations:
    "B-Protein":    "B-PROTEIN",
    "I-Protein":    "I-PROTEIN",
}

def parse_conll_file(filepath):
    """
    Parse a plain CoNLL-format file (token<whitespace>tag per line, 
    blank line between sentences).
    Returns list of (tokens, labels) tuples.
    """
    sentences = []
    tokens, labels = [], []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()
            if line == "" or line.startswith("-DOCSTART-"):
                if tokens:
                    sentences.append((tokens, labels))
                    tokens, labels = [], []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    tokens.append(parts[0])
                    labels.append(parts[-1])  # last column = tag
    if tokens:
        sentences.append((tokens, labels))
    return sentences


def parse_csv_ner(filepath, word_col="Word", tag_col="Tag", sent_col=None):
    """
    Parse a CSV-format NER dataset.
    Sentences are delimited either by a sentence-id column or by NaN rows.
    """
    df = pd.read_csv(filepath, encoding="unicode_escape")
    print(f"  CSV columns: {list(df.columns)}")

    # Try to identify columns automatically
    word_col  = next((c for c in df.columns if c.lower() in ["word", "token", "words"]), word_col)
    tag_col   = next((c for c in df.columns if c.lower() in ["tag", "label", "ner", "bio"]), tag_col)
    sent_col  = next((c for c in df.columns if "sent" in c.lower()), sent_col)

    print(f"  Using: word='{word_col}', tag='{tag_col}', sent='{sent_col}'")

    sentences = []
    tokens, labels = [], []

    for _, row in df.iterrows():
        word = row.get(word_col)
        tag  = row.get(tag_col)

        # A new sentence starts when the sentence-id column has a value
        # (only filled on the first token of each sentence in many Kaggle CSVs)
        if sent_col and pd.notna(row.get(sent_col)) and tokens:
            sentences.append((tokens, labels))
            tokens, labels = [], []

        if pd.notna(word) and pd.notna(tag):
            tokens.append(str(word))
            labels.append(str(tag))

    if tokens:
        sentences.append((tokens, labels))

    return sentences


def load_genia(genia_dir):
    """
    Auto-detect file format and load GENIA sentences.
    """
    genia_dir = Path(genia_dir)

    # Look for CoNLL-style text files first
    txt_files = list(genia_dir.rglob("*.txt")) + list(genia_dir.rglob("*.conll"))
    csv_files = list(genia_dir.rglob("*.csv"))

    if txt_files:
        print(f"Parsing CoNLL files: {txt_files}")
        sentences = []
        for f in txt_files:
            sentences.extend(parse_conll_file(f))
        return sentences

    elif csv_files:
        print(f"Parsing CSV files: {csv_files}")
        sentences = []
        for f in csv_files:
            sentences.extend(parse_csv_ner(f))
        return sentences

    else:
        raise FileNotFoundError(f"No .txt, .conll, or .csv files found in {genia_dir}")


genia_sentences = load_genia(GENIA_DIR)
print(f"\nLoaded {len(genia_sentences)} GENIA sentences")
# Preview first sentence
print("\nExample sentence:")
print(list(zip(*genia_sentences[0])))

In [ ]:
def apply_label_map(sentences, label_map):
    """
    Remap labels in a list of (tokens, labels) tuples.
    Unknown labels are kept as-is with a warning.
    """
    unknown = set()
    remapped = []
    for tokens, labels in sentences:
        new_labels = []
        for l in labels:
            mapped = label_map.get(l)
            if mapped is None:
                unknown.add(l)
                mapped = l  # keep original
            new_labels.append(mapped)
        remapped.append((tokens, new_labels))
    if unknown:
        print(f"  ⚠ Unmapped labels (kept as-is): {unknown}")
    return remapped


def save_conll_from_tuples(sentences, output_path):
    """Save a list of (tokens, labels) tuples to CoNLL format."""
    with open(output_path, "w", encoding="utf-8") as f:
        for tokens, labels in tqdm(sentences, desc=f"Writing {Path(output_path).name}"):
            for token, label in zip(tokens, labels):
                f.write(f"{token}\t{label}\n")
            f.write("\n")
    print(f"  Saved {len(sentences)} sentences → {output_path}")


# Apply label mapping
genia_sentences = apply_label_map(genia_sentences, GENIA_LABEL_MAP)

# Split into train / val / test (80 / 10 / 10) with fixed seed
train_val, test   = train_test_split(genia_sentences, test_size=0.10, random_state=SEED)
train,     val    = train_test_split(train_val,        test_size=0.11, random_state=SEED)  # ~10% of total

print(f"GENIA split → train: {len(train)}, val: {len(val)}, test: {len(test)}")

save_conll_from_tuples(train, BASE_OUT/"ner/genia_train.conll")
save_conll_from_tuples(val,   BASE_OUT/"ner/genia_val.conll")
save_conll_from_tuples(test,  BASE_OUT/"ner/genia_test.conll")

print("\nGENIA ✓")

---
## 3. Merge NER Datasets

Combine BC5CDR + GENIA into one merged training set.  
Val and test sets are kept **separate per dataset** so performance can be evaluated per-source.  
Output: `ner_train_merged.conll`

In [ ]:
def read_conll_file(filepath):
    """Read a CoNLL file back into (tokens, labels) tuples."""
    sentences = []
    tokens, labels = [], []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()
            if line == "":
                if tokens:
                    sentences.append((tokens, labels))
                    tokens, labels = [], []
            else:
                parts = line.split("\t")
                tokens.append(parts[0])
                labels.append(parts[1] if len(parts) > 1 else "O")
    if tokens:
        sentences.append((tokens, labels))
    return sentences


bc5cdr_train = read_conll_file(BASE_OUT/"ner/bc5cdr_train.conll")
genia_train  = read_conll_file(BASE_OUT/"ner/genia_train.conll")

merged = bc5cdr_train + genia_train
random.shuffle(merged)  # shuffle at sentence level

save_conll_from_tuples(merged, BASE_OUT/"ner/ner_train_merged.conll")
print(f"\nMerged training set: {len(bc5cdr_train)} (BC5CDR) + {len(genia_train)} (GENIA) = {len(merged)} sentences")

# Collect all unique labels in the merged set
all_labels = sorted({l for _, labels in merged for l in labels})
print(f"All labels in merged training set: {all_labels}")

# Save label list as JSON for teammates to import
with open(BASE_OUT/"ner/label_list.json", "w") as f:
    json.dump(all_labels, f, indent=2)
print(f"Label list saved to {BASE_OUT}/ner/label_list.json")

---
## 4. DrugBank — Relation Extraction Dataset

**Source:** Kaggle `sergeguillemart/drugbank` or `devildev89/drug-bank-5110`  
**Goal:** Extract drug–disease pairs and create sentence-level training examples  
**Output:** `relations/drugbank_relations.csv`

Columns: `sentence`, `drug`, `disease`, `relation`  

> Since DrugBank gives structured pairs (no natural sentences), we build template sentences.  
> Entity spans are marked with `[E1]...[/E1]` and `[E2]...[/E2]` for the RE model.

In [ ]:
# ── Inspect available DrugBank files ─────────────────────────────────────────
# Try both common Kaggle dataset slugs
DRUGBANK_CANDIDATES = [
    Path("/kaggle/input/drugbank"),
    Path("/kaggle/input/drug-bank-5110"),
    Path("/kaggle/input/drugbank5110"),
]

DRUGBANK_DIR = None
for candidate in DRUGBANK_CANDIDATES:
    if candidate.exists():
        DRUGBANK_DIR = candidate
        break

if DRUGBANK_DIR is None:
    raise FileNotFoundError(
        "DrugBank folder not found. Make sure you added the dataset to this Kaggle notebook.\n"
        "Expected one of: " + str(DRUGBANK_CANDIDATES)
    )

print(f"DrugBank directory: {DRUGBANK_DIR}")
print("Files:")
for f in DRUGBANK_DIR.rglob("*"):
    print(" ", f, f"({f.stat().st_size // 1024} KB)" if f.is_file() else "")

In [ ]:
# ── Load the drugs CSV and inspect columns ────────────────────────────────────
# DrugBank CSVs commonly have these column patterns — we auto-detect them.

csv_files = list(DRUGBANK_DIR.rglob("*.csv"))
print(f"CSV files found: {csv_files}")

# Load the first (or main) CSV
main_csv = csv_files[0]
df_drug = pd.read_csv(main_csv, low_memory=False)
print(f"\nShape: {df_drug.shape}")
print(f"Columns: {list(df_drug.columns)}")
df_drug.head(3)

In [ ]:
# ── Extract drug-disease pairs from indication/description text ───────────────

# Common column name variants across DrugBank Kaggle datasets
def find_column(df, candidates):
    for c in candidates:
        matches = [col for col in df.columns if c.lower() in col.lower()]
        if matches:
            return matches[0]
    return None

name_col       = find_column(df_drug, ["name", "drug_name", "common_name"])
indication_col = find_column(df_drug, ["indication", "description", "pharmacodynamics"])

print(f"Drug name column   : '{name_col}'")
print(f"Indication column  : '{indication_col}'")

if name_col is None or indication_col is None:
    print("\nColumn auto-detection failed. Available columns:")
    print(df_drug.columns.tolist())
    print("\nPlease manually set name_col and indication_col below.")
    # ↓ Uncomment and set manually if needed:
    # name_col       = "Name"
    # indication_col = "Indication"

In [ ]:
# ── Disease extraction from indication text ───────────────────────────────────
# Pattern: "for the treatment of X", "indicated for X", "used in X", etc.

TREATMENT_PATTERNS = [
    r"(?:for\s+(?:the\s+)?treatment\s+of)\s+([^.;,\(]{3,80})",
    r"(?:indicated\s+for(?:\s+the\s+treatment\s+of)?)\s+([^.;,\(]{3,80})",
    r"(?:used\s+to\s+treat)\s+([^.;,\(]{3,80})",
    r"(?:for\s+use\s+in)\s+([^.;,\(]{3,80})",
    r"(?:management\s+of)\s+([^.;,\(]{3,80})",
]

SIDE_EFFECT_PATTERNS = [
    r"(?:may\s+cause)\s+([^.;,\(]{3,60})",
    r"(?:associated\s+with)\s+([^.;,\(]{3,60})",
    r"(?:can\s+cause)\s+([^.;,\(]{3,60})",
]


def extract_pairs(drug_name, indication_text):
    """
    Extract (drug, disease, relation) triples from indication text.
    Returns a list of dicts.
    """
    if pd.isna(indication_text) or len(str(indication_text).strip()) < 10:
        return []

    text   = str(indication_text).strip()
    drug   = str(drug_name).strip()
    pairs  = []

    for pattern in TREATMENT_PATTERNS:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            disease = match.group(1).strip().rstrip(".")
            disease = re.sub(r"\s+", " ", disease)  # normalize whitespace
            if 3 < len(disease) < 80:                # sanity length filter
                pairs.append({"drug": drug, "disease": disease, "relation": "TREATS"})

    for pattern in SIDE_EFFECT_PATTERNS:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            symptom = match.group(1).strip().rstrip(".")
            symptom = re.sub(r"\s+", " ", symptom)
            if 3 < len(symptom) < 60:
                pairs.append({"drug": drug, "disease": symptom, "relation": "CAUSES"})

    return pairs


all_pairs = []
for _, row in tqdm(df_drug.iterrows(), total=len(df_drug), desc="Extracting drug-disease pairs"):
    all_pairs.extend(extract_pairs(row[name_col], row[indication_col]))

df_pairs = pd.DataFrame(all_pairs).drop_duplicates()
print(f"\nExtracted {len(df_pairs)} drug-disease pairs")
print(df_pairs["relation"].value_counts())
df_pairs.head()

In [ ]:
# ── Build template sentences with entity markers ──────────────────────────────
# Entity markers tell the RE model where E1 (drug) and E2 (disease) are.
# Format: [E1] drug [/E1] ... [E2] disease [/E2]

SENTENCE_TEMPLATES = {
    "TREATS": [
        "[E1] {drug} [/E1] is indicated for the treatment of [E2] {disease} [/E2] .",
        "[E1] {drug} [/E1] is used to treat [E2] {disease} [/E2] .",
        "[E1] {drug} [/E1] is prescribed for [E2] {disease} [/E2] .",
        "Patients with [E2] {disease} [/E2] are treated with [E1] {drug} [/E1] .",
    ],
    "CAUSES": [
        "[E1] {drug} [/E1] may cause [E2] {disease} [/E2] .",
        "[E1] {drug} [/E1] has been associated with [E2] {disease} [/E2] .",
        "Administration of [E1] {drug} [/E1] can lead to [E2] {disease} [/E2] .",
    ],
}

def build_sentence(row):
    templates = SENTENCE_TEMPLATES.get(row["relation"], SENTENCE_TEMPLATES["TREATS"])
    # Rotate templates to create variety (use drug name hash to deterministically pick)
    idx = hash(row["drug"]) % len(templates)
    return templates[idx].format(drug=row["drug"], disease=row["disease"])

df_pairs["sentence"] = df_pairs.apply(build_sentence, axis=1)

# Reorder columns for clarity
df_pairs = df_pairs[["sentence", "drug", "disease", "relation"]]

# Split into train / val / test
train_re, temp  = train_test_split(df_pairs, test_size=0.20, random_state=SEED, stratify=df_pairs["relation"])
val_re,   test_re = train_test_split(temp,   test_size=0.50, random_state=SEED, stratify=temp["relation"])

print(f"Relation splits → train: {len(train_re)}, val: {len(val_re)}, test: {len(test_re)}")

train_re.to_csv(BASE_OUT/"relations/drugbank_train.csv",  index=False)
val_re.to_csv(  BASE_OUT/"relations/drugbank_val.csv",    index=False)
test_re.to_csv( BASE_OUT/"relations/drugbank_test.csv",   index=False)
df_pairs.to_csv(BASE_OUT/"relations/drugbank_all.csv",    index=False)  # full set

print("\nDrugBank ✓")
df_pairs.head()

---
## 5. MTSamples — Unlabeled Clinical Transcriptions

**Source:** Kaggle `tboyle10/medicaltranscriptions`  
**Goal:** Clean medical transcription text → sentence-segmented plain text  
**Output:** `unlabeled/mtsamples_sentences.txt` (one sentence per line)  
  
Used by the ASR demo teammate as realistic clinical domain text.

In [ ]:
MTSAMPLES_PATH = Path("/kaggle/input/medicaltranscriptions/mtsamples.csv")

df_mt = pd.read_csv(MTSAMPLES_PATH)
print(f"MTSamples shape: {df_mt.shape}")
print(f"Columns: {list(df_mt.columns)}")
print(f"Medical specialties ({df_mt['medical_specialty'].nunique()}): {df_mt['medical_specialty'].value_counts().head(10).to_dict()}")
df_mt.head(2)

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

# ── Cleaning helpers ──────────────────────────────────────────────────────────

def clean_transcription(text):
    """
    Clean a raw MTSamples transcription:
    - Remove de-identification markers like [** ... **]
    - Remove section headers (ALL CAPS lines)
    - Normalize whitespace
    - Remove lines that are purely numeric or too short
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove de-id markers: [**Name**], [**2020-01-01**], etc.
    text = re.sub(r"\[\*\*.*?\*\*\]", "[REDACTED]", text)

    # Remove lines that are section headers (all caps, short)
    lines = text.split("\n")
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        # Skip all-caps header lines (e.g. "HISTORY OF PRESENT ILLNESS:")
        if stripped.isupper() and len(stripped) < 80:
            continue
        # Skip very short lines (likely artifacts)
        if len(stripped) < 15:
            continue
        cleaned_lines.append(stripped)

    text = " ".join(cleaned_lines)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def segment_sentences(text, min_length=20, max_length=512):
    """Use spaCy to sentence-tokenize cleaned clinical text."""
    if not text or len(text) < min_length:
        return []
    doc = nlp(text)
    sentences = []
    for sent in doc.sents:
        s = sent.text.strip()
        if min_length <= len(s) <= max_length:
            sentences.append(s)
    return sentences


# ── Process all transcriptions ────────────────────────────────────────────────
all_sentences = []

for _, row in tqdm(df_mt.iterrows(), total=len(df_mt), desc="Processing MTSamples"):
    cleaned = clean_transcription(row.get("transcription"))
    sentences = segment_sentences(cleaned)
    all_sentences.extend(sentences)

# Deduplicate
all_sentences = list(dict.fromkeys(all_sentences))

print(f"\nTotal clean sentences extracted: {len(all_sentences)}")
print("\nSample sentences:")
for s in all_sentences[:3]:
    print(f"  · {s}")

In [ ]:
# Save one sentence per line
out_path = BASE_OUT / "unlabeled/mtsamples_sentences.txt"
with open(out_path, "w", encoding="utf-8") as f:
    for sent in all_sentences:
        f.write(sent + "\n")

print(f"Saved {len(all_sentences)} sentences → {out_path}")

# Also save a specialty-filtered version (e.g. Cardiology only)
# Useful if your ASR demo teammate wants domain-specific data
specialty_map = {}
for _, row in df_mt.iterrows():
    specialty = str(row.get("medical_specialty", "Unknown")).strip()
    cleaned   = clean_transcription(row.get("transcription"))
    sents     = segment_sentences(cleaned)
    specialty_map.setdefault(specialty, []).extend(sents)

for specialty, sents in specialty_map.items():
    safe_name = re.sub(r"[^a-zA-Z0-9_]", "_", specialty).lower()
    out = BASE_OUT / f"unlabeled/mtsamples_{safe_name}.txt"
    with open(out, "w", encoding="utf-8") as f:
        for s in sents:
            f.write(s + "\n")

print(f"\nSpecialty-specific files saved: {len(specialty_map)} files")
print("\nMTSamples ✓")

---
## 6. Final Verification

Quick sanity checks — confirm files exist and have the right format.

In [ ]:
print("=" * 60)
print("OUTPUT FILE SUMMARY")
print("=" * 60)

for f in sorted(BASE_OUT.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size // 1024
        print(f"  {f.relative_to(BASE_OUT)}  ({size_kb} KB)")

print()

# ── CoNLL format check ────────────────────────────────────────────────────────
def count_conll_stats(filepath):
    sents, tokens, entity_counts = 0, 0, {}
    with open(filepath) as f:
        for line in f:
            line = line.rstrip()
            if line == "":
                sents += 1
            else:
                parts = line.split("\t")
                tokens += 1
                if len(parts) > 1 and parts[1] != "O":
                    entity_counts[parts[1]] = entity_counts.get(parts[1], 0) + 1
    return sents, tokens, entity_counts

print("NER FILE STATS:")
for conll_file in sorted((BASE_OUT/"ner").glob("*.conll")):
    sents, tokens, entities = count_conll_stats(conll_file)
    top_entities = sorted(entities.items(), key=lambda x: -x[1])[:4]
    print(f"  {conll_file.name}: {sents} sentences, {tokens} tokens, top entities: {top_entities}")

print()
print("RELATION FILE STATS:")
for csv_file in sorted((BASE_OUT/"relations").glob("*.csv")):
    df = pd.read_csv(csv_file)
    print(f"  {csv_file.name}: {len(df)} rows, labels: {df['relation'].value_counts().to_dict()}")

print()
print("UNLABELED FILE STATS:")
main_txt = BASE_OUT/"unlabeled/mtsamples_sentences.txt"
with open(main_txt) as f:
    lines = f.readlines()
print(f"  mtsamples_sentences.txt: {len(lines)} sentences")
avg_len = sum(len(l.split()) for l in lines) / max(len(lines), 1)
print(f"  Average sentence length: {avg_len:.1f} words")

print()
print("All preprocessing complete! Share the processed_data/ folder with your team.")

---
## 7. Upload to Google Drive (Optional)

Run this cell to push `processed_data/` directly to a shared Google Drive folder.

> **Alternatively:** Download `processed_data.zip` from the Kaggle output panel and upload manually.

In [ ]:
# ── Zip the output folder for easy download ───────────────────────────────────
import shutil

zip_path = "/kaggle/working/processed_data"
shutil.make_archive(zip_path, "zip", BASE_OUT)
print(f"Created: {zip_path}.zip")
print("Download it from the Kaggle output panel → processed_data.zip")

# ── OR: Mount Google Drive and copy ──────────────────────────────────────────
# Uncomment the lines below if you want to push directly to Drive:

# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copytree(str(BASE_OUT), "/content/drive/MyDrive/clinical_nlp/processed_data", dirs_exist_ok=True)
# print("Uploaded to Google Drive.")